5.5   Tagging Unknown Words <br/>
Our approach to tagging unknown words still uses backoff to a regular-expression tagger or a default tagger. These are unable to make use of context. Thus, if our tagger encountered the word blog, not seen during training, it would assign it the same tag, regardless of whether this word appeared in the context the blog or to blog. How can we do better with these unknown words, or out-of-vocabulary items?

A useful method to tag unknown words based on context is to limit the vocabulary of a tagger to the most frequent n words, and to replace every other word with a special word UNK using the method shown in 3. During training, a unigram tagger will probably learn that UNK is usually a noun. However, the n-gram taggers will detect contexts in which it has some other tag. For example, if the preceding word is to (tagged TO), then UNK will probably be tagged as a verb.

In [34]:
from nltk.corpus import wordnet
from fuzzywuzzy import fuzz, StringMatcher
import csv
import nltk
import re
import string
import pandas as pd
import zipfile
import numpy as np
from nltk.tokenize import word_tokenize
from nltk.tag import pos_tag
from nltk.corpus import brown
from nltk.corpus import gutenberg
from collections import defaultdict

In [102]:
tagged = brown.tagged_sents(categories = 'news')
size = int(len(tagged)*.9)
train = tagged[:size]
test = tagged[size:]

#let n be 2000 most freq words
#doing what was told above
#increasing n might incease accuracy a little bit
fd = nltk.FreqDist(brown.words(categories='news'))
most_freq = fd.most_common(2000)
length = len(train)
mark = 0
train=list(train)
for x in range(length) :
    len2 = len(train[x])
    for x1 in range(len2):
        mark = 0
        for (word2,_) in most_freq:
            if(train[x][x1][0] ==  word2 ):
                mark=1
                break
        if(mark == 0):
            t=np.array(train[x][x1])
            t[0]='UNK'
            train[x][x1]=tuple(t)


#Taggers
t0 = nltk.DefaultTagger('NN')
t1 = nltk.UnigramTagger(train,backoff=t0)
t2 = nltk.BigramTagger(train, backoff = t1)
t3 = nltk.TrigramTagger(train, backoff = t2 )
t3.evaluate(test)

0.7779328216884281